# 📓 Semana 5 · Dia 2 — Structured Streaming: batch vs streaming

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA, DEP (streaming) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Streaming simples rodando com watermark |

---


## 📖 Teoria — Batch vs Streaming

**Batch**: processa um lote de dados já completo (ex.: diário).
**Streaming**: processa dados que chegam continuamente (ex.: eventos em tempo real), em micro-batches.

O **Structured Streaming** do Spark usa a **mesma API de DataFrame** — a diferença é `readStream`/`writeStream` + checkpoint.


## 📖 Teoria — Conceitos-chave

- **trigger**: quando o micro-batch roda (`once`, `processingTime`, `continuous`)
- **watermark**: tolerância para dados atrasados (ex.: `watermark('ts', '1 hour')`)
- **checkpointLocation**: onde o estado é persistido — sem isso, o stream não recupera após falha
- **outputMode**: `append` (novas linhas), `update` (linhas atualizadas), `complete` (agregações totais)


### 💻 Na prática — Streaming simples

Leia o landing como stream e escreva em Delta com checkpoint.


In [ ]:
# Ler o CSV como stream
stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/vol_checkpoints/schema_stream")
    .option("header", True)
    .load("/Volumes/workspace/bronze/vol_landing"))
print("Stream criado (lazy).")

In [ ]:
# Escrever com trigger once (para estudo)
(stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/bronze/vol_checkpoints/ckpt_stream")
    .outputMode("append")
    .trigger(once=True)
    .table("workspace.bronze.vendas_stream"))
print("Streaming batch-único executado.")

### 💻 Na prática — Watermark e agregações em stream

Streaming com agregação por janela de tempo.


In [ ]:
# Agregação com janela temporal + watermark
from pyspark.sql.functions import window, sum as s, to_timestamp
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/vol_checkpoints/schema_win")
    .option("header", True)
    .load("/Volumes/workspace/bronze/vol_landing")
    .withColumn("ts", to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))
    .withWatermark("ts", "1 hour")
    .groupBy(window("ts", "1 day"), "Country")
    .agg(s("Quantity").alias("qtd")))
print("Stream com janela diária e watermark de 1h configurado.")

In [ ]:
# Escrever o resultado (append)
q = (df.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/bronze/vol_checkpoints/ckpt_win")
    .outputMode("append")
    .trigger(once=True)
    .table("workspace.bronze.vendas_janela"))
q.awaitTermination()
print("Agregação em stream concluída.")

> 🎯 **Dica de prova**: Watermark + janela é pergunta clássica de streaming (DEP). Memorize: watermark define o atraso tolerado; janela agrupa por tempo.


## 🎯 Exercícios de fixação

**1.** Explique a diferença entre trigger(once=True) e processingTime.

**2.** O que acontece sem checkpointLocation?

**3.** Qual outputMode usar para agregação com atualizações?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Triggers

once: processa tudo que chegou e para (batch incremental). processingTime: roda a cada N segundos continuamente (stream de verdade).

**2.** Sem checkpoint

O stream não consegue recuperar estado após falha — pode reprocessar (perde exactly-once) ou falhar ao reiniciar.

**3.** OutputMode

`update` — emite somente linhas que mudaram; ideal para agregações com atualização contínua. `append` não permite atualização.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*